# Inter-judge agreement

Cohen's kappa between the primary judge (Llama-3.1-8B-Instruct) and each additional judge (GPT-5.1, Falcon-7B-Instruct) on the directional closer-reference decision.


In [ ]:
import os, sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import numpy as np
import pandas as pd

from src.conflicts import get_conflict
from src.statistics.aggregation import load_run_scores
from src.translator.prompts.prompts import SYSTEM_PROMPTS_LIB


## Config


In [ ]:
CONFLICT = 'ru_ua'
RESULTS_ROOT = '../../results'
JUDGE_PROMPT = 'primary'

PRIMARY_JUDGE = 'llama3-8b'
ADDITIONAL_JUDGES = ['gpt-5.1', 'falcon-7b-instruct']

MODELS = [
    'mistral-7b-instruct-v0.3', 'qwen25-7b', 'llama3-8b',
    'gemini-3-pro-preview', 'deepseek-7b-chat', 'falcon-7b-instruct',
    'moonshot-v1-8k', 'gpt-5.1',
]
PROMPTS = list(SYSTEM_PROMPTS_LIB)


## Collect directional labels per judge


In [ ]:
conflict = get_conflict(CONFLICT)
side_a, side_b = conflict.sides

def directional_labels(judge_label):
    labels = {}
    for model in MODELS:
        for prompt in PROMPTS:
            a = load_run_scores(results_root=RESULTS_ROOT, conflict=conflict, judge_label=judge_label, judge_prompt=JUDGE_PROMPT, translator_label=model, prompt_name=prompt, side=side_a)
            b = load_run_scores(results_root=RESULTS_ROOT, conflict=conflict, judge_label=judge_label, judge_prompt=JUDGE_PROMPT, translator_label=model, prompt_name=prompt, side=side_b)
            for i in sorted(set(a) & set(b)):
                key = f'{model}/{prompt}/{i}'
                if a[i] < b[i]:
                    labels[key] = side_a.code
                elif b[i] < a[i]:
                    labels[key] = side_b.code
                else:
                    labels[key] = 'tie'
    return labels

primary = directional_labels(PRIMARY_JUDGE)
print(f'primary judge {PRIMARY_JUDGE}: {len(primary)} decisions')


## Cohen's kappa against each additional judge


In [ ]:
def cohens_kappa(a, b):
    ids = [k for k in a if k in b and a[k] != 'tie' and b[k] != 'tie']
    if not ids:
        return {'n': 0, 'kappa': None, 'agreement': None}
    cats = sorted({a[k] for k in ids} | {b[k] for k in ids})
    idx = {c: i for i, c in enumerate(cats)}
    M = np.zeros((len(cats), len(cats)))
    for k in ids:
        M[idx[a[k]], idx[b[k]]] += 1
    n = M.sum()
    po = M.trace() / n
    pe = ((M.sum(0) / n) * (M.sum(1) / n)).sum()
    kappa = (po - pe) / (1 - pe) if pe < 1 else 0.0
    return {'n': int(n), 'agreement': float(po), 'kappa': float(kappa)}

rows = []
for judge in ADDITIONAL_JUDGES:
    alt = directional_labels(judge)
    rows.append({'judge': judge, **cohens_kappa(primary, alt)})

pd.DataFrame(rows)
